In [1]:
import pandas as pd
from cdc_ml.config import PREFERENCE_EXTERNAL,RECORDS_PROCESSED,BOOKING_CYCLES_PROCESSED,RECORDS_PROCESSED
from cdc_ml.datasets.preference.preference import clean_df

2026-05-08 23:03:06.119 | INFO     | cdc_ml.config:<module>:12 - PROJ_ROOT path is: C:\Users\zhiju\Desktop\cdc_ml


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SLOT_ORDER = ["t_0830", "t_1020", "t_1245", "t_1435", "t_1625", "t_1850", "t_2040"]
WEEK_ORDER = ["mon", "tue", "wed", "thu", "fri", "sat", "sun"]


def _melt_pref(df: pd.DataFrame) -> pd.DataFrame:
    out = (
        df.melt(
            id_vars=["username", "class_type", "is_one_team", "day_of_week"],
            value_vars=SLOT_ORDER,
            var_name="slots",
            value_name="available",
        )
        .query("available == 1")
        .drop(columns="available")
    )
    out["day_of_week"] = out["day_of_week"].map(dict(enumerate(WEEK_ORDER)))
    return out


def plot_pref_heatmaps(df_pref_pv: pd.DataFrame, group_col: str):
    """Faceted slot × day heatmaps of unique-user counts, one panel per group value."""
    counts = (
        df_pref_pv
        .groupby(["day_of_week", "slots", group_col])["username"]
        .nunique()
        .reset_index()
    )

    groups = sorted(counts[group_col].unique())
    fig, axes = plt.subplots(1, len(groups), figsize=(5 * len(groups), 5), sharey=True)
    axes = np.atleast_1d(axes)
    vmax = counts["username"].max()

    for ax, g in zip(axes, groups):
        pv_raw = (
        counts[counts[group_col] == g]
        .pivot(index="slots", columns="day_of_week", values="username")
        )
        pv = (
            counts[counts[group_col] == g]
            .pivot(index="slots", columns="day_of_week", values="username")
            .reindex(index=SLOT_ORDER, columns=WEEK_ORDER)
            .fillna(0).astype(int)
        )
        sns.heatmap(
            pv, annot=True, fmt="d", cmap="Blues",
            vmin=0, vmax=vmax, ax=ax, cbar=(ax is axes[-1]),
        )
        ax.set_title(f"{group_col} = {g}")
        ax.set_xlabel("")
        ax.set_ylabel("")

    fig.suptitle(f"Unique users available by slot × day, split by {group_col}")
    plt.tight_layout()
    plt.show()


def plot_pref_proportion(df_pref_pv: pd.DataFrame, binary_col: str = "is_one_team"):
    """Single heatmap: proportion of available users in each (slot, day) cell with binary_col == 1."""
    prop = (
        df_pref_pv
        .groupby(["day_of_week", "slots"])[binary_col]
        .mean()
        .reset_index()
        .pivot(index="slots", columns="day_of_week", values=binary_col)
        .reindex(index=SLOT_ORDER, columns=WEEK_ORDER)
    )

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(
        prop, annot=True, fmt=".0%", cmap="RdBu_r",
        center=0.5, vmin=0, vmax=1, ax=ax,
    )
    ax.set_title(f"Share of {binary_col} = 1 by slot × day")
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()


def plot_pref_bars(df_pref_pv: pd.DataFrame, group_col: str = "is_one_team"):
    """Faceted bar chart: unique-user counts per slot, one panel per day, hue = group."""
    counts = (
        df_pref_pv
        .groupby(["day_of_week", "slots", group_col])["username"]
        .nunique()
        .reset_index()
    )
    g = sns.catplot(
        data=counts, kind="bar",
        x="slots", y="username", hue=group_col,
        col="day_of_week", col_order=WEEK_ORDER, col_wrap=4,
        order=SLOT_ORDER, height=3, aspect=1.2,
    )
    g.set_xticklabels(rotation=45)
    g.set_titles("{col_name}")
    g.figure.suptitle(f"Unique users by slot, split by {group_col}", y=1.02)
    plt.show()


def get_pref_pivot(df: pd.DataFrame):
    df_pref_pv = _melt_pref(df)

    plot_pref_heatmaps(df_pref_pv, group_col="is_one_team")
    plot_pref_heatmaps(df_pref_pv, group_col="class_type")
    plot_pref_proportion(df_pref_pv, binary_col="is_one_team")
    plot_pref_bars(df_pref_pv, group_col="is_one_team")  # uncomment if you want the bar view too

    return df_pref_pv


df_pref_pv = get_pref_pivot(df_pref)